<a href="https://colab.research.google.com/github/mdrs-thiago/mdrs-thiago.github.io/blob/main/topicos_a/2027/notebooks/lab02_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Lab 2 — Fine-tuning de CNN pré-treinada
**Tópicos Especiais A · Deep Learning · UERJ 2027.1** · Aula 08 · Bloco 2 — Redes Convolucionais

**Objetivo:** comparar treino do zero, feature extraction e fine-tuning completo de uma ResNet-18 no Oxford-IIIT Pets.

| | |
|---|---|
| ✅ **Checkpoint (em sala)** | 60 XP: veja as marcações `✅ CHECKPOINT` |
| 📦 **Entrega (7 dias)** | 180 XP: tabela das 3 estratégias, curvas, matriz de confusão e 5 erros comentados. |

> Preencha apenas os trechos marcados com `# TODO`. Antes de começar: **Arquivo → Salvar uma cópia no Drive** e, se indicado, **Ambiente de execução → Alterar tipo → T4 GPU**.

## 0. Setup (ligue a GPU!)

In [ ]:
import random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

In [ ]:
import time, copy, torch.nn as nn, matplotlib.pyplot as plt, pandas as pd
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
from torchvision.transforms import v2

## 1. Dados e augmentation
Traga o pipeline do **Desafio 7**. Atenção: validação **sem** aleatoriedade e com a normalização do ImageNet.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# TODO: substitua pelo seu pipeline do Desafio 7
train_tfm = v2.Compose([
    v2.RandomResizedCrop(224, scale=(0.7, 1.0)), v2.RandomHorizontalFlip(),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])
val_tfm = v2.Compose([v2.Resize(256), v2.CenterCrop(224),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])

tr_aug = datasets.OxfordIIITPet('data', split='trainval', download=True, transform=train_tfm)
tr_plain = datasets.OxfordIIITPet('data', split='trainval', download=True, transform=val_tfm)
test_ds = datasets.OxfordIIITPet('data', split='test', download=True, transform=val_tfm)
idx = torch.randperm(len(tr_aug), generator=torch.Generator().manual_seed(42))
n_val = 600
train_ds, val_ds = Subset(tr_aug, idx[n_val:].tolist()), Subset(tr_plain, idx[:n_val].tolist())
CLASSES = tr_aug.classes; N_CLS = len(CLASSES)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=128, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=2)
print(len(train_ds), len(val_ds), len(test_ds), N_CLS)

## 2. Treino genérico (adaptado do Lab 1, com AMP para acelerar na GPU)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    tot, ls, ok = 0, 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with torch.autocast(device_type=device, enabled=(device == 'cuda')):
                logits = model(xb); loss = loss_fn(logits, yb)
            if train:
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            ls += loss.item() * len(yb); ok += (logits.argmax(1) == yb).sum().item(); tot += len(yb)
    return ls / tot, ok / tot

def train_model(model, opt, epochs=8):
    hist, best, best_state, t0 = {'val_acc': []}, 0, None, time.time()
    for ep in range(epochs):
        tl, ta = run_epoch(model, train_loader, opt)
        vl, va = run_epoch(model, val_loader)
        hist['val_acc'].append(va)
        print(f'ep {ep+1} | train {tl:.3f}/{ta:.3f} | val {vl:.3f}/{va:.3f}')
        if va > best: best, best_state = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); hist['tempo_s'] = time.time() - t0
    return hist

n_trainable = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. (a) CNN pequena do zero

In [ ]:
def small_cnn(n_cls):
    # TODO: adapte a SmallCNN da Aula 5 para entrada 3×224×224.
    # Dica: 4 blocos [Conv(3x3) → BN → ReLU → MaxPool(2)] com 32, 64, 128, 256 canais,
    # depois nn.AdaptiveAvgPool2d(1) → Flatten → Linear(256, n_cls)
    return ...

set_seed(42)
m_a = small_cnn(N_CLS).to(device)
h_a = train_model(m_a, torch.optim.AdamW(m_a.parameters(), lr=1e-3))

## 4. (b) Feature extraction: ResNet-18 congelada + nova cabeça
### ✅ CHECKPOINT: (b) deve superar (a) na validação.

In [ ]:
set_seed(42)
m_b = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
# TODO: congele todos os parâmetros e troque m_b.fc por uma Linear com N_CLS saídas
...
m_b = m_b.to(device)
print('treináveis:', n_trainable(m_b))
h_b = train_model(m_b, torch.optim.AdamW([p for p in m_b.parameters() if p.requires_grad], lr=1e-3))

## 5. (c) Fine-tuning completo com LRs discriminativos

In [ ]:
set_seed(42)
m_c = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
m_c.fc = nn.Linear(m_c.fc.in_features, N_CLS)
m_c = m_c.to(device)
backbone = [p for n, p in m_c.named_parameters() if not n.startswith('fc')]
# TODO: AdamW com dois grupos: backbone lr=1e-4, cabeça (m_c.fc) lr=1e-3; weight_decay=1e-2
opt_c = ...
print('treináveis:', n_trainable(m_c))
h_c = train_model(m_c, opt_c)

## 6. Comparação

In [ ]:
rows = []
for name, m, h in [('(a) do zero', m_a, h_a), ('(b) feature extraction', m_b, h_b), ('(c) fine-tuning', m_c, h_c)]:
    rows.append({'estratégia': name, 'treináveis': n_trainable(m), 'melhor acc val': max(h['val_acc']), 'tempo (s)': round(h['tempo_s'])})
    plt.plot(h['val_acc'], label=name)
plt.legend(); plt.title('acurácia de validação'); plt.show()
pd.DataFrame(rows)

## 7. Matriz de confusão e erros do melhor modelo

In [ ]:
from sklearn.metrics import confusion_matrix
best = m_c  # TODO: troque se outro for melhor na validação
best.eval(); ys, ps, xs = [], [], []
with torch.no_grad():
    for xb, yb in val_loader:
        p = best(xb.to(device)).argmax(1).cpu()
        ys += yb.tolist(); ps += p.tolist(); xs += list(xb)
cm = confusion_matrix(ys, ps)
np.fill_diagonal(cm, 0)
top = np.dstack(np.unravel_index(np.argsort(cm.ravel())[::-1][:3], cm.shape))[0]
for i, j in top: print(f'{CLASSES[i]} → {CLASSES[j]}: {cm[i, j]} vezes')

In [ ]:
def show(x):
    x = x.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    return x.clip(0, 1)
errs = [k for k in range(len(ys)) if ys[k] != ps[k]][:5]
fig, axs = plt.subplots(1, 5, figsize=(16, 4))
for ax, k in zip(axs, errs):
    ax.imshow(show(xs[k])); ax.axis('off')
    ax.set_title(f'real: {CLASSES[ys[k]]}\npred: {CLASSES[ps[k]]}', fontsize=8)

## 8. Análise (📦 entrega)
- Por que (b) é tão melhor que (a) com tão poucos parâmetros treináveis?
- (c) valeu o custo extra em relação a (b)?
- Comente os 5 erros: são "razoáveis" (raças parecidas) ou indicam um problema?
- Acurácia de teste do melhor modelo (avalie **uma vez**): ...